In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

In [ ]:
class BeerDataset(Dataset):
    def __init__(
        self,
        meta_csv: str | Path = "data/meta/bing.csv",
        root: str | Path | None = None,
        transform=None,
        target_transform=None,
        return_info: bool = False,
    ):
        self.meta_csv = Path(meta_csv)
        if not self.meta_csv.exists():
            raise FileNotFoundError(f"Metadata CSV not found: {self.meta_csv}")

        self.root = Path(root) if root is not None else Path('.')
        self.transform = transform
        self.target_transform = target_transform
        self.return_info = return_info

        df = pd.read_csv(self.meta_csv)
        required_cols = {"orig_path", "klass"}
        missing = required_cols - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns in metadata: {sorted(missing)}")

        df = df.dropna(subset=["orig_path", "klass"]).reset_index(drop=True)
        self._records = df
        classes = sorted(df["klass"].unique())
        self.class_to_idx = {klass: idx for idx, klass in enumerate(classes)}
        self.idx_to_class = {idx: klass for klass, idx in self.class_to_idx.items()}

    def __len__(self) -> int:
        return len(self._records)

    def _resolve_path(self, rel_path: str) -> Path:
        p = Path(rel_path)
        if p.is_absolute():
            return p
        return (self.root / p).resolve()

    def __getitem__(self, index: int):
        row = self._records.iloc[index]
        img_path = self._resolve_path(row["orig_path"])
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found on disk: {img_path}")

        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[row["klass"]]

        if self.transform is not None:
            image = self.transform(image)
        if self.target_transform is not None:
            label = self.target_transform(label)

        if self.return_info:
            info = row.to_dict() | {"img_path": str(img_path)}
            return image, label, info
        return image, label


In [ ]:
from skimage import io

